### Oncology Project

## 1. Introduction

This notebook performs exploratory data analysis (EDA) on SEER Research Data for Stage III colon cancer (primary site C18.0–C18.9). The goal is to understand the structure, quality, and distributions of the extracted case listing before it's used to fit and compare survival models (Cox Proportional Hazards vs. Random Survival Forest).

**Purpose of this notebook:** Data quality checks, missingness, distribution of key covariates (age, stage, follow-up time), and censoring patterns — before any modeling begins.

------

In [1]:
#import standard libraies
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


## Cohort Definition

**Population:** Stage III colon cancer (AJCC Stage IIIA, IIIB, IIIC, III, IIINOS), primary site C18.0–C18.9, diagnosis years 2010–2023, sourced from SEER Research Data via SEER*Stat (Case Listing Session export).

**Endpoint:** Overall survival (OS) — event defined as death from any cause, derived from `Vital status recode (study cutoff used)`. Cause-specific survival was intentionally not used, to avoid cause-of-death attribution ambiguity and keep this notebook's downstream Cox vs. RSF comparison clean.

**Staging:** AJCC stage is consolidated from three year-specific SEER fields spanning 7th edition (2010–2015), a 7th-edition recode covering the 2016–2017 gap, and 8th edition / EOD 2018 fields (2018+). See `derive_stage_2018()` and `consolidate_stage()` in `cleaning.py` for the full mapping logic and code-level rationale.

**Source of truth:** All cleaning, staging derivation, missing-value handling, and cohort filtering logic lives in `src/data/cleaning.py`, version-controlled and covered by `tests/software/test_cleaning.py`. This notebook loads the *output* of that pipeline (`data/processed/cleaned_data.csv`) — it does not re-derive cleaning decisions inline. Any discrepancy between what's explored here and what `cleaning.py` actually does should be treated as a bug in one of the two, not resolved by patching around it in the notebook.

In [4]:
BASE_DIR = os.path.dirname(os.getcwd())
data_folder = os.path.join(BASE_DIR, "data")
file = os.path.join(data_folder, "processed", "cleaned_data.csv")
df = pd.read_csv(file)
print(f"Data loaded successfully")
print(df.head())

Data loaded successfully
   Patient ID Site recode ICD-O-3/WHO 2008  Year of diagnosis  \
0        9246                Sigmoid Colon               2011   
1       14329                        Cecum               2013   
2       52005                        Cecum               2019   
3       91345                        Cecum               2017   
4       95479                        Cecum               2017   

    Sequence number                 Type of Reporting Source  \
0  One primary only  Hospital inpatient/outpatient or clinic   
1  One primary only  Hospital inpatient/outpatient or clinic   
2  One primary only  Hospital inpatient/outpatient or clinic   
3  One primary only  Hospital inpatient/outpatient or clinic   
4  One primary only  Hospital inpatient/outpatient or clinic   

  Behavior recode for analysis Age recode with <1 year olds and 90+     Sex  \
0                    Malignant                          50-54 years  Female   
1                    Malignant           